# DataVine Analytics: Machine Learning Applications

## Project Overview

DataVine Analytics is a consulting group that develops machine learning solutions for organizations across different industries. This project applies a complete machine learning workflow to three different business problems involving classification, recommendation, and clustering.

The first project focuses on a **Wine Classification System**, where a premium wine distributor needs to automatically classify wine varieties based on their chemical properties. A k-Nearest Neighbors (k-NN) classification model will be developed and optimized using Principal Component Analysis (PCA) and hyperparameter tuning.

The second project develops an **Agricultural Feed Recommendation Engine** using the `Chickwts` dataset as a proxy for feed performance. PCA and cosine similarity will be used to identify feed types with similar performance characteristics and generate recommendations.

The third project addresses **Regional Crime Pattern Analysis** using the `USArrests` dataset. K-Means and Gaussian Mixture Model (GMM) clustering will be used to identify natural groupings among regions based on crime statistics. PCA will also be used to reduce the dimensionality of the selected features and visualize the resulting clusters.

Across all three projects, the analysis follows a structured machine learning workflow: data preparation, exploratory analysis, feature scaling, dimensionality reduction, model development, hyperparameter optimization where appropriate, evaluation, visualization, and interpretation.

The goal is not only to build technically appropriate machine learning models, but also to translate their results into meaningful insights that could support business and organizational decision-making.

## Project Roadmap

This notebook is organized into three independent machine learning applications, each addressing a different type of analytical problem.

| Project | Business Problem | Machine Learning Approach | Main Output |
|---|---|---|---|
| Wine Classification | Automatically identify wine varieties | PCA + k-NN + GridSearchCV | Predicted wine class |
| Feed Recommendation | Identify similar-performing feed types | PCA + Cosine Similarity | Similar feed recommendations |
| Crime Pattern Analysis | Identify regions with similar crime patterns | PCA + K-Means + GMM | Regional clusters |

Although the three projects use different machine learning techniques, they follow the same underlying workflow:

**Understand the data → Prepare the data → Transform the features → Apply the appropriate technique → Evaluate the results → Interpret the findings**

In [ ]:
#import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.pipeline import Pipeline

# 1. Dataset Preparation

Before applying any machine learning technique, the datasets must be inspected and prepared to ensure that the models receive reliable and appropriately formatted inputs.

The three datasets represent different analytical problems and therefore have different structures. The Wine dataset contains chemical measurements and a categorical wine class, the Chickwts dataset contains feed types and chicken weight measurements, and the USArrests dataset contains regional crime statistics.

The preparation stage will focus on:

- Loading each dataset correctly
- Inspecting the number of observations and variables
- Identifying numerical and categorical variables
- Checking for missing values
- Checking for duplicate or inconsistent records
- Reviewing descriptive statistics
- Identifying potential data quality issues
- Standardizing numerical features where required

The objective is to understand the characteristics of each dataset before selecting and applying the appropriate machine learning technique.

In [ ]:
# Load each CSV dataset into a separate DataFrame.

wine = pd.read_csv("wine.csv")
chickwts = pd.read_csv("chickwts.csv")
us_arrests = pd.read_csv("USArrests.csv")

In [ ]:
# Display the first five rows of each dataset
# to understand the variables and the way the data is structured.

print("WINE DATASET")
display(wine.head())

print("\nCHICKWTS DATASET")
display(chickwts.head())

print("\nUSARRESTS DATASET")
display(us_arrests.head())

WINE DATASET


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0



CHICKWTS DATASET


,weight,feed
0,179,horsebean
1,160,horsebean
2,136,horsebean
3,227,horsebean
4,217,horsebean



USARRESTS DATASET


,rownames,Murder,Assault,UrbanPop,Rape
0,Alabama,13.2,236,58,21.2
1,Alaska,10.0,263,48,44.5
2,Arizona,8.1,294,80,31.0
3,Arkansas,8.8,190,50,19.5
4,California,9.0,276,91,40.6


In [ ]:
# Check the number of rows and columns in each dataset.

print("Wine shape:", wine.shape)
print("Chickwts shape:", chickwts.shape)
print("USArrests shape:", us_arrests.shape)

Wine shape: (178, 14)
Chickwts shape: (71, 2)
USArrests shape: (50, 5)


In [ ]:
# Inspect data types and identify potential formatting issues.

print("WINE DATASET")
wine.info()

print("\nCHICKWTS DATASET")
chickwts.info()

print("\nUSARRESTS DATASET")
us_arrests.info()

WINE DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       178 non-null    float64
 1   malic_acid                    178 non-null    float64
 2   ash                           178 non-null    float64
 3   alcalinity_of_ash             178 non-null    float64
 4   magnesium                     178 non-null    float64
 5   total_phenols                 178 non-null    float64
 6   flavanoids                    178 non-null    float64
 7   nonflavanoid_phenols          178 non-null    float64
 8   proanthocyanins               178 non-null    float64
 9   color_intensity               178 non-null    float64
 10  hue                           178 non-null    float64
 11  od280/od315_of_diluted_wines  178 non-null    float64
 12  proline                       178 non-null    float

In [ ]:
# Check for missing values and duplicate rows in each dataset.

datasets = {
    "Wine": wine,
    "Chickwts": chickwts,
    "USArrests": us_arrests
}

for name, data in datasets.items():
    print(f"\n{name} Dataset")
    print("-" * 30)
    print("Missing values:")
    print(data.isnull().sum())
    print("Duplicate rows:", data.duplicated().sum())


Wine Dataset
------------------------------
Missing values:
alcohol                         0
malic_acid                      0
ash                             0
alcalinity_of_ash               0
magnesium                       0
total_phenols                   0
flavanoids                      0
nonflavanoid_phenols            0
proanthocyanins                 0
color_intensity                 0
hue                             0
od280/od315_of_diluted_wines    0
proline                         0
target                          0
dtype: int64
Duplicate rows: 0

Chickwts Dataset
------------------------------
Missing values:
weight    0
feed      0
dtype: int64
Duplicate rows: 1

USArrests Dataset
------------------------------
Missing values:
rownames    0
Murder      0
Assault     0
UrbanPop    0
Rape        0
dtype: int64
Duplicate rows: 0


In [ ]:
# Generate descriptive statistics for the numerical variables
# in each dataset.

print("WINE DATASET")
display(wine.describe())

print("\nCHICKWTS DATASET")
display(chickwts.describe())

print("\nUSARRESTS DATASET")
display(us_arrests.describe())

WINE DATASET


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
count,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000
mean,13.000618,2.336348,2.366517,19.494944,99.741573,2.295112,2.029270,0.361854,1.590899,5.058090,0.957449,2.611685,746.893258,0.938202
std,0.811827,1.117146,0.274344,3.339564,14.282484,0.625851,0.998859,0.124453,0.572359,2.318286,0.228572,0.709990,314.907474,0.775035
min,11.030000,0.740000,1.360000,10.600000,70.000000,0.980000,0.340000,0.130000,0.410000,1.280000,0.480000,1.270000,278.000000,0.000000
25%,12.362500,1.602500,2.210000,17.200000,88.000000,1.742500,1.205000,0.270000,1.250000,3.220000,0.782500,1.937500,500.500000,0.000000
50%,13.050000,1.865000,2.360000,19.500000,98.000000,2.355000,2.135000,0.340000,1.555000,4.690000,0.965000,2.780000,673.500000,1.000000
75%,13.677500,3.082500,2.557500,21.500000,107.000000,2.800000,2.875000,0.437500,1.950000,6.200000,1.120000,3.170000,985.000000,2.000000
max,14.830000,5.800000,3.230000,30.000000,162.000000,3.880000,5.080000,0.660000,3.580000,13.000000,1.710000,4.000000,1680.000000,2.000000



CHICKWTS DATASET


,weight
count,71.000000
mean,261.309859
std,78.073700
min,108.000000
25%,204.500000
50%,258.000000
75%,323.500000
max,423.000000



USARRESTS DATASET


,Murder,Assault,UrbanPop,Rape
count,50.00000,50.000000,50.000000,50.000000
mean,7.78800,170.760000,65.540000,21.232000
std,4.35551,83.337661,14.474763,9.366385
min,0.80000,45.000000,32.000000,7.300000
25%,4.07500,109.000000,54.500000,15.075000
50%,7.25000,159.000000,66.000000,20.100000
75%,11.25000,249.000000,77.750000,26.175000
max,17.40000,337.000000,91.000000,46.000000


In [ ]:
# Check the number of observations in each wine class.
# This helps us understand whether the classification target is balanced.

print("Wine target distribution:")
print(wine["target"].value_counts().sort_index())


# Check how many observations belong to each feed type.
# This helps us understand the representation of each feed
# before calculating similarities.

print("\nChickwts feed distribution:")
print(chickwts["feed"].value_counts())


# Inspect the region names stored in the identifier column.

print("\nUSArrests regions:")
print(us_arrests["rownames"].tolist())

Wine target distribution:
target
0    59
1    71
2    48
Name: count, dtype: int64

Chickwts feed distribution:
feed
soybean      14
linseed      12
sunflower    12
casein       12
meatmeal     11
horsebean    10
Name: count, dtype: int64

USArrests regions:
['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming']


In [ ]:
# Display the duplicate row(s) in the Chickwts dataset.
# This allows us to determine whether the duplicate represents
# a genuine repeated observation or a possible data-entry issue.

chickwts[chickwts.duplicated(keep=False)]

,weight,feed
24,248,soybean
35,248,soybean


In [ ]:
# Remove duplicate rows from the Chickwts dataset.
# keep="first" keeps the first occurrence and removes subsequent duplicates.

chickwts = chickwts.drop_duplicates()

In [ ]:
print("Chickwts shape after removing duplicates:", chickwts.shape)
print("Duplicate rows:", chickwts.duplicated().sum())

Chickwts shape after removing duplicates: (70, 2)
Duplicate rows: 0


## 1.3 Data Quality Findings

The initial data-quality assessment shows that the three datasets are generally complete and suitable for further analysis.

- The **Wine dataset** contains no missing values and no duplicate records.
- The **Chickwts dataset** contains no missing values but initially contained one duplicate record. The duplicate consisted of two identical observations with a weight of 248 and a feed type of soybean. One duplicate record was removed to prevent the same observation from being counted twice.
- The **USArrests dataset** contains no missing values or duplicate records.

The datasets therefore require minimal cleaning. The main preparation steps will focus on appropriate feature selection and standardization before applying PCA, classification, recommendation, and clustering techniques.

Importantly, standardization will be performed within each project rather than modifying the original datasets during the general data-preparation stage.

# 2. Wine Classification System

## 2.1 Business Problem

DataVine Analytics has been asked to develop an automated wine classification system for a premium wine distributor. The objective is to classify wine varieties based on their chemical properties.

Accurate classification can support inventory management and quality control by providing a data-driven method for identifying wine varieties from their chemical measurements.

A **k-Nearest Neighbors (k-NN)** classification model will be developed, with **Principal Component Analysis (PCA)** used to reduce dimensionality while retaining 95% of the information in the original features. GridSearchCV will then be used to identify suitable k-NN hyperparameters.

In [ ]:
# Separate the chemical features from the target variable.

X = wine.drop(columns="target")
y = wine["target"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Target classes:", sorted(y.unique()))

Feature shape: (178, 13)
Target shape: (178,)
Target classes: [0, 1, 2]


In [ ]:
# Split the data into training and testing sets.
# stratify=y maintains the proportion of wine classes in both sets.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (142, 13)
Testing set: (36, 13)


In [ ]:
# Create a pipeline that standardizes the features and then applies PCA.
# PCA will retain enough components to explain 95% of the variance.

wine_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),
    ("knn", KNeighborsClassifier())
])

# Fit the preprocessing and model on the training data only.
wine_pipeline.fit(X_train, y_train)

# Check how many principal components were required.
n_components = wine_pipeline.named_steps["pca"].n_components_

print("Original number of features:", X_train.shape[1])
print("Number of PCA components:", n_components)

Original number of features: 13
Number of PCA components: 10
